<a href="https://colab.research.google.com/github/kuds/rl-drone/blob/main/notebooks/%5BDrone%20Racer%5D%20Soft%20Actor-Critic%20(SAC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Drone Racer] Soft Actor-Critic (SAC)

Train a Soft Actor-Critic agent on `DroneRacerEnv` to follow a randomly
generated 3D circular race track. Checkpoints are placed equidistantly
around a circle of configurable radius, then radially perturbed and lifted
off the ground plane to produce a smooth 3D spline. The drone must hit each
checkpoint in order, or it pays a heavy penalty.

| Field | Value |
| --- | --- |
| Environment | `DroneRacerEnv` (3D circular spline track) |
| Algorithm | Soft Actor-Critic (SAC, off-policy, continuous control) |
| Policy | `MlpPolicy` with frame-stacking (4) |
| Default total steps | 10,000,000 |
| Episode length | 5,000 steps |
| Checkpoints per track | 12 |
| Vectorized envs | 4 |
| Observation / reward normalization | `VecNormalize` |
| Reward function | `multiplicative_inverse` |

## What this notebook does

1. Installs `rl-drone[train,curve]` (the `curve` extra pulls in the spline
   fitter), configures EGL, and verifies GPU access.
2. Builds the artifact directory tree for the `DroneRacer` run.
3. Generates a sample noisy circular track, fits a `Curve3D` spline through
   it, and visualizes the result so you can sanity-check the geometry.
4. Trains SAC on `DroneRacerEnv` for ten million steps with the full
   callback stack.
5. Reloads the best model, evaluates it, records **five** independent
   playback videos (each on a freshly randomized track), plots the learning
   curves, and snapshots a copy of the notebook into the run folder.


## 1. Environment Setup & Dependencies

Install `rl-drone[train,curve]` directly from GitHub (the `curve` extra adds
SciPy + the spline fitter used to build race tracks), configure the NVIDIA
EGL ICD so MuJoCo can render off-screen on the Colab GPU, verify that
MuJoCo compiles a trivial XML model, then install `ffmpeg` + `mediapy` for
video recording and clear the install spam from the output area.


In [ ]:
!pip install "rl-drone[train,curve] @ git+https://github.com/kuds/rl-drone.git"

# Set up GPU rendering.
import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy

from IPython.display import clear_output
clear_output()

## 2. Imports

Bring in the standard library, NumPy / Torch, the `rl_drone` racer
environment and helpers (track generation, `Curve3D` spline fitting,
plotting, run-paths, callbacks, summary writer), and the Stable-Baselines3
SAC pieces.


In [ ]:
# --- Standard Library ---
import os
import platform
import shutil
from datetime import datetime
from importlib.metadata import version

# --- Third-Party ---
import numpy as np
import torch

# --- rl_drone package ---
from rl_drone.envs.drone_racer import DroneRacerEnv
from rl_drone.utils.model_xml import setup_mujoco_model
from rl_drone.utils.track import (
    generate_equidistant_points,
    add_radial_noise_to_points_rng,
)
from rl_drone.utils.curve import Curve3D
from rl_drone.utils.plotting import plot_learning_curves, plot_training_reward_over_time
from rl_drone.utils.paths import build_run_paths
from rl_drone.utils.summary import write_stage_summary
from rl_drone.callbacks import (
    ConfigSaveCallback,
    ReformatEvalCallback,
    TrainingPlotsCallback,
    VecNormalizeSaveCallback,
    VideoRecordCallback,
)

# --- Stable Baselines3 ---
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    CallbackList,
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    VecNormalize,
    VecVideoRecorder,
)

# --- Environment Specific (Colab) ---
try:
    import google.colab.drive
except ImportError:
    pass  # Gracefully handle if not running in Colab

np.set_printoptions(precision=3, suppress=True, linewidth=100)

## 3. Verify Library Versions

Print the versions of every key dependency (Python, Torch, CUDA, Gymnasium,
NumPy, MuJoCo, Stable-Baselines3, and `rl-drone`) so the run is reproducible
and any future regressions are easy to diagnose.


In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Robot Description Version: {version('robot_descriptions')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Mujoco Version: {version('mujoco')}")
print(f"Stable-Baselines3 Version: {version('stable-baselines3')}")
print(f"Matplotlib Version: {version('matplotlib')}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
if torch.cuda.is_available(): print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 4. Run Configuration & Artifact Layout

Pick the algorithm (`SAC`) and environment string (`DroneRacer`), then call
`build_run_paths` to produce a unique timestamped directory tree (Google
Drive by default) for every artifact this run will create. All later cells
use the returned `paths` object so models, plots, checkpoints, videos, and
TensorBoard logs end up neatly grouped under one folder per run.


In [ ]:
rl_type = "SAC"
project_name = "BitCrazy"
env_str = "DroneRacer"
name_prefix = f"{env_str}_{rl_type}".lower()

use_google_drive = True
parent_path = None
if use_google_drive:
    parent_path = "/content/gdrive"
    google.colab.drive.mount(parent_path, force_remount=True)

paths = build_run_paths(
    project_name=project_name,
    env_str=env_str,
    rl_type=rl_type,
    parent_path=parent_path,
    use_google_drive=use_google_drive,
)

# Backwards-compatible aliases used throughout the notebook
log_dir = paths.base_dir
model_folder_path = paths.run_dir

## 5. Hyperparameters

Two dictionaries fully describe this run:

* **`env_config`** — racer-specific parameters: frame-stacking depth, track
  geometry (radius, number of checkpoints, height), episode length, the
  reward function name, the no-contact early-termination threshold, and all
  reward shaping terms (time penalty, out-of-bounds penalty, completion
  bonus, contact bonus, ...).
* **`model_config`** — SAC training parameters: evaluation frequency, number
  of parallel envs, total training steps, and number of evaluation episodes.

Keeping the two cleanly separated makes it easy to sweep one without
disturbing the other, and both dictionaries are persisted to disk by
`ConfigSaveCallback`.


In [ ]:
env_config = {
    "frame_stack": 4,
    "episode_length": 5_000,
    "sphere_size": 0.1,
    "track_size": 1,
    "number_of_checkpoints": 12,
    "reward_function": "multiplicative_inverse",
    "terminate_without_contact": 450,
    "speed_factor": 0.25,
    "track_height": 0.25,
    "max_distance": 0.5,
    "time_penalty": 0.1,
    "out_of_bounds_penalty": -100,
    "no_contact_penalty": -100,
    "complete_bonus": 250,
    "contact_bonus": 25
}

model_config = {
    "eval_freq": 25_000,
    "n_envs": 4,
    "total_timesteps": 10_000_000,
    "n_eval_episodes": 25
}

In [ ]:
# Create artifact directories for this run
paths.makedirs()

## 6. Configure the MuJoCo Drone Model

Edit the bundled MuJoCo XML model in place to apply the chosen sphere size
and target height for the racer track. This must run **before** any
environment is constructed because the env constructor reads the model from
disk.


In [ ]:
setup_mujoco_model(
    sphere_size=env_config["sphere_size"],
    target_height=env_config["track_height"],
)

## 7. Sample Track Generation & Visualization

Generate a single example track to verify the geometry helpers behave as
expected: lay 12 equidistant points on a circle, perturb them radially and
in `z`, fit a closed `Curve3D` spline through the noisy points, and plot the
result. Each training episode samples a fresh track from the same
distribution, so this cell is just a *visualization* — not part of the
training loop.


In [ ]:
# --- Track generation and visualization ---
CIRCLE_RADIUS = env_config["track_size"]
NUM_POINTS = env_config["number_of_checkpoints"]
TRACK_HEIGHT = env_config["track_height"]
z_noise_scale = 0.25
angle_noise_scale = 0.25

equidistant_points = generate_equidistant_points(CIRCLE_RADIUS, NUM_POINTS, TRACK_HEIGHT)

print(f"Generated {NUM_POINTS} equidistant points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(equidistant_points):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f})")

noisy_coords = add_radial_noise_to_points_rng(
    equidistant_points, angle_noise_scale, z_noise_scale, skip_origin=True, seed=0
)
print(f"Generated {NUM_POINTS} equidistant noisy points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(noisy_coords):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f}, Z={p[2]:.2f})")

noisy_curve = Curve3D(noisy_coords, closed=True, num_samples=200)

print(f"\nCreated curve through {len(noisy_coords)} points")
print(f"Curve length: {noisy_curve.get_length():.4f}")
print(f"Curve is closed: {noisy_curve.closed}")

noisy_curve.visualize()

## 8. Inspect the Environment

Quick smoke test: instantiate `DroneRacerEnv(env_config)` once, print its
observation and action spaces, and tear it back down. This catches schema
mismatches before kicking off the long training run.


In [ ]:
env = DroneRacerEnv(env_config)
print("Observation Space Size: ", env.observation_space.shape)
print('Actions Space: ', env.action_space)
env.close()

## 9. Vectorized Environment Factory

Define the `make_env` factory used by `make_vec_env` to spin up parallel
training and evaluation workers. Every constructed env reuses the
`env_config` from above and runs `check_env` so any Gym API violations fail
fast.


In [ ]:
def make_env():
  env = DroneRacerEnv(render_mode="rgb_array", env_config=env_config)
  check_env(env)
  return env

## 10. Train the SAC Agent

Build the training and evaluation `VecNormalize` wrappers, register the
full callback stack — checkpoints, periodic evaluation, CSV reformatting,
config snapshot, learning-curve plots, and video capture — and call
`model.learn()` for ten million steps. After training, the script saves the
final model along with the `VecNormalize` statistics for both train and
eval envs.

> Ten million steps is a long training run. The callback stack is
> deliberately verbose so that even a stopped run still leaves useful
> artifacts on disk for inspection or resumption.


In [ ]:
# Create Training environment
env = make_vec_env(make_env,
                   n_envs=model_config["n_envs"],
                   monitor_dir=paths.monitor_dir)

normalized_env = VecNormalize(env,
                              training=True,
                              norm_obs=True,
                              norm_reward=True)


# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1)
normalized_env_val = VecNormalize(env_val,
                                  training=False,
                                  norm_obs=True,
                                  norm_reward=True)

checkpoint_callback = CheckpointCallback(
  save_freq=model_config["eval_freq"],
  save_path=paths.checkpoints_dir,
  name_prefix="rl_model",
  save_replay_buffer=False,
  save_vecnormalize=True,
)

reformat_callback = ReformatEvalCallback(
    save_path=paths.run_dir,
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    save_freq=model_config["eval_freq"],
    csv_file_name=name_prefix,
)

vec_normalize_callback = VecNormalizeSaveCallback(save_path=paths.run_dir)

config_save_callback = ConfigSaveCallback(
    save_path=paths.run_dir,
    hyperparams={"env_config": env_config, "model_config": model_config},
    run_name=f"{env_str}_{rl_type}",
)

training_plots_callback = TrainingPlotsCallback(
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    save_path=paths.plots_dir,
    save_freq=model_config["eval_freq"],
    name_prefix=name_prefix,
)

video_record_callback = VideoRecordCallback(
    make_env_fn=make_env,
    save_path=paths.videos_dir,
    video_length=10_000,
    save_freq=model_config["eval_freq"],
    name_prefix=name_prefix,
)

eval_callback = EvalCallback(normalized_env_val,
                             best_model_save_path=paths.run_dir,
                             log_path=paths.run_dir,
                             render=False,
                             deterministic=True,
                             callback_on_new_best=vec_normalize_callback,
                             n_eval_episodes=model_config["n_eval_episodes"],
                             eval_freq=model_config["eval_freq"])

# Create the callback list
callbackList = CallbackList([
    checkpoint_callback,
    eval_callback,
    reformat_callback,
    config_save_callback,
    training_plots_callback,
    video_record_callback,
])

# learning with tensorboard logging and saving model
model = SAC("MlpPolicy",
            normalized_env,
            verbose=0,
            tensorboard_log=paths.tensorboard_dir)

training_start_time = datetime.now()

model.learn(total_timesteps=model_config["total_timesteps"],
            callback=callbackList,
            progress_bar=False)
training_end_time = datetime.now()

# Save the model
model.save(os.path.join(paths.run_dir, "final_model"))

mean_reward, std_reward = evaluate_policy(model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Final Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

normalized_env.save(os.path.join(paths.run_dir, "final_normalized_env.pkl"))
normalized_env_val.save(os.path.join(paths.run_dir, "final_normalized_env_val.pkl"))

env.close()
env_val.close()
normalized_env.close()
normalized_env_val.close()

## 11. Evaluate the Best Model

Reload `best_model.zip` (saved by `EvalCallback`) together with its
matching `VecNormalize` statistics, run a deterministic evaluation across
`n_eval_episodes` random tracks, and write a human-readable stage-summary
report to disk.


In [ ]:
# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1, seed=0)
normalized_env_val = VecNormalize.load(os.path.join(paths.run_dir, "best_model_vec_normalize.pkl"),
                                       venv=env_val)

# Load the best model
best_model_path = os.path.join(paths.run_dir, "best_model")
best_model = SAC.load(best_model_path, env=normalized_env_val)

mean_reward, std_reward = evaluate_policy(best_model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")


# Write a human-readable stage summary report.
write_stage_summary(
    save_dir=paths.run_dir,
    model=best_model,
    eval_env=normalized_env_val,
    project_name=project_name,
    env_str=env_str,
    algorithm=rl_type,
    total_timesteps=model_config["total_timesteps"],
    training_start=training_start_time,
    training_end=training_end_time,
    description="SAC training run for drone racer.",
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    n_eval_episodes=30,
)

# Close the environment
env_val.close()
normalized_env_val.close()

## 12. Record Playback Videos

Record **five** independent rollout videos of the best model. Each
iteration spins up a fresh vectorized env (so the track is re-randomized),
loads the same best model + `VecNormalize` stats, plays the policy
deterministically until the episode terminates, and saves the resulting
MP4 to the run's `videos/` folder. Total episode reward and step count are
printed for each playback.


In [ ]:
# Record video of the best model
video_length = 10_000
os.makedirs(paths.videos_dir, exist_ok=True)
for i in range(5):
    env_val = make_vec_env(make_env, n_envs=1)
    normalized_env_val = VecNormalize.load(
        os.path.join(paths.run_dir, "best_model_vec_normalize.pkl"),
        venv=env_val,
    )

    best_model_path = os.path.join(paths.run_dir, "best_model")
    best_model = SAC.load(best_model_path, env=normalized_env_val)
    best_model_file_name = f"best_model_{name_prefix}_{i}"
    env = VecVideoRecorder(
        normalized_env_val,
        paths.videos_dir,
        video_length=video_length,
        record_video_trigger=lambda x: x == 0,
        name_prefix=best_model_file_name,
    )

    obs = env.reset()
    total_reward = 0.0
    session_length = 0
    for _ in range(video_length):
        action, _states = best_model.predict(obs, deterministic=True)
        obs, rewards, done, info = env.step(action)
        total_reward += env.get_original_reward()
        env.render()
        session_length += 1
        if done:
            break

    print(f"Playback {i+1}: steps={session_length}, reward={total_reward[0]:.2f}")

    env.close()
    env_val.close()
    normalized_env_val.close()

## 13. Plot Learning Curves

Render two complementary views of reward over time:

1. **Evaluation learning curves** — reward and episode length at the
   discrete eval checkpoints saved in `evaluations.npz` by
   `EvalCallback`.
2. **Training reward over time** — the dense per-episode reward trace
   read from the SB3 `Monitor` CSVs in `paths.monitor_dir`, with a
   rolling mean overlaid to make the learning trend easy to read.

The same curves are also auto-saved during training by
`TrainingPlotsCallback` — this final plot is just a convenient summary
at the end of the run.

In [ ]:
# Evaluation learning curves from evaluations.npz
eval_file = os.path.join(paths.run_dir, "evaluations.npz")
plot_learning_curves(
    eval_file,
    save_dir=paths.plots_dir,
    name_prefix=name_prefix,
    show=True,
)

# Dense training-reward-over-time trace from SB3 Monitor CSVs
plot_training_reward_over_time(
    paths.monitor_dir,
    smoothing_window=50,
    title=f"{rl_type} Training Reward over Time",
    save_path=os.path.join(
        paths.plots_dir, f"{name_prefix}_training_reward_over_time.png"
    ),
    show=True,
)

print(f"Training plots also auto-saved during training to: {paths.plots_dir}")

## 14. Snapshot the Notebook

Use Colab's `%notebook` magic to dump a copy of the live notebook (with
outputs!) into the run folder. This makes every run completely
self-describing — code, configuration, results, and plots all live in the
same timestamped directory.


In [ ]:
# Save a copy of this notebook to the run folder
notebook_name = "notebook.ipynb"
%notebook -e $notebook_name

source_file = os.path.join(notebook_name)
destination_file = os.path.join(paths.run_dir, notebook_name)

try:
    shutil.copyfile(notebook_name, destination_file)
    print(f"File '{source_file}' copied to '{destination_file}' successfully.")
except FileNotFoundError:
    print(f"Error: Source file '{source_file}' not found.")
except shutil.SameFileError:
    print(f"Error: Source and destination are the same file.")
except Exception as e:
    print(f"An error occurred: {e}")

## 15. Disconnect the Colab Runtime

Free the Colab GPU after the run completes so we don't keep burning compute
quota while the artifacts upload to Google Drive.


In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime
runtime.unassign()